# Pruning

## Learning Objectives
1. Understand how magnitude-based pruning removes low-weight parameters to induce sparsity.
2. Implement structured and unstructured pruning with `torch.nn.utils.prune` and handle OOM gracefully.
3. Apply iterative lottery-ticket pruning to find a sparse subnetwork that matches dense accuracy.
4. Compare sparsity levels against accuracy and latency to determine the efficiency-accuracy frontier.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Magnitude Pruning — NumPy Sparsity

In [ ]:
# Demonstrate unstructured magnitude pruning on a weight matrix using only numpy.
# Weights with absolute value below a threshold are set to zero (pruned).

def magnitude_prune(weights: np.ndarray, sparsity: float) -> np.ndarray:
    """Zero out the lowest-magnitude weights to achieve target sparsity.
    
    Args:
        weights: 2-D weight matrix (out_features x in_features).
        sparsity: Fraction of weights to zero out, e.g. 0.5 = 50% sparse.
    Returns:
        Pruned weight matrix with same shape.
    """
    flat = weights.ravel()
    # Threshold: keep only top-(1-sparsity) fraction by magnitude
    threshold = np.percentile(np.abs(flat), sparsity * 100)
    mask = np.abs(weights) >= threshold
    return weights * mask


rng = np.random.default_rng(42)
W = rng.standard_normal((64, 128))  # Simulate a linear layer weight matrix

for sparsity in [0.0, 0.25, 0.50, 0.75, 0.90]:
    W_pruned = magnitude_prune(W, sparsity)
    actual = np.mean(W_pruned == 0)
    # Frobenius-norm ratio shows how much weight magnitude was retained
    norm_ratio = np.linalg.norm(W_pruned) / np.linalg.norm(W)
    print(f"Target sparsity {sparsity:.0%} | Actual {actual:.1%} zeros | "
          f"Weight norm ratio {norm_ratio:.3f}")

## Level 2: PyTorch Structured & Unstructured Pruning with OOM Handling

In [ ]:
# Build a small MLP, apply both unstructured (element-wise) and structured
# (channel-wise L2) pruning via torch.nn.utils.prune.

class MLP(nn.Module):
    def __init__(self, in_features: int = 128, hidden: int = 256, out: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, out)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))


def count_sparsity(model: nn.Module) -> float:
    """Return fraction of zero parameters across all linear layers."""
    total = zeros = 0
    for module in model.modules():
        if isinstance(module, nn.Linear):
            w = module.weight.data
            zeros += (w == 0).sum().item()
            total += w.numel()
    return zeros / total if total > 0 else 0.0


# --- Synthetic data ---
X_data = torch.randn(1000, 128, device=device)
y_data = torch.randint(0, 10, (1000,), device=device)
train_loader = DataLoader(
    TensorDataset(X_data[:800], y_data[:800]), batch_size=64, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(X_data[800:], y_data[800:]), batch_size=64
)


def quick_train(model: nn.Module, epochs: int = 5) -> float:
    """Train model briefly and return validation accuracy."""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            try:
                opt.zero_grad()
                crit(model(xb), yb).backward()
                opt.step()
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print("OOM — reduce batch size or model size")
                    torch.cuda.empty_cache()
                    continue
                raise
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb).argmax(1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


# Train baseline
model_base = MLP().to(device)
acc_base = quick_train(model_base, epochs=10)
print(f"Baseline | sparsity={count_sparsity(model_base):.1%} | val_acc={acc_base:.3f}")

# --- Unstructured pruning: zero out 50% of smallest weights globally ---
import copy
model_unst = copy.deepcopy(model_base)
parameters_to_prune = [
    (model_unst.fc1, 'weight'),
    (model_unst.fc2, 'weight'),
    (model_unst.fc3, 'weight'),
]
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.5,
)
# Make mask permanent
for module, name in parameters_to_prune:
    prune.remove(module, name)
print(f"Unstructured 50% | sparsity={count_sparsity(model_unst):.1%}")

# --- Structured pruning: remove 20% of channels (rows) per layer by L2 norm ---
model_struct = copy.deepcopy(model_base)
for module in [model_struct.fc1, model_struct.fc2]:
    prune.ln_structured(module, name='weight', amount=0.2, n=2, dim=0)
    prune.remove(module, 'weight')
print(f"Structured 20%   | sparsity={count_sparsity(model_struct):.1%}")

## Real-World Example 1: Iterative Lottery Ticket Pruning

In [ ]:
# Lottery Ticket Hypothesis: iteratively prune then retrain from original init.
# Each round: train -> prune 20% -> reset surviving weights to init values.

import copy

def lottery_ticket_round(
    model: nn.Module,
    init_state: dict,
    prune_rate: float = 0.20,
    epochs: int = 5,
) -> tuple:
    """One lottery-ticket round: train -> prune -> reset to init.
    
    Returns (model_after_reset, val_accuracy, sparsity).
    """
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

    # Prune globally by magnitude
    params_to_prune = [(m, 'weight') for m in model.modules()
                       if isinstance(m, nn.Linear)]
    prune.global_unstructured(
        params_to_prune, pruning_method=prune.L1Unstructured, amount=prune_rate
    )
    # Capture mask BEFORE removing it
    masks = {id(m): getattr(m, 'weight_mask').clone() for m, _ in params_to_prune}
    for m, name in params_to_prune:
        prune.remove(m, name)

    # Reset surviving weights to original initialisation values
    for m, _ in params_to_prune:
        m.weight.data = init_state[id(m)] * masks[id(m)]

    sparsity = count_sparsity(model)

    # Quick val accuracy
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return model, correct / total, sparsity


# Store initial weights
lottery_model = MLP().to(device)
init_state = {id(m): m.weight.data.clone()
              for m in lottery_model.modules() if isinstance(m, nn.Linear)}

print("Round | Sparsity | Val Acc")
print("-" * 35)
for rnd in range(4):
    lottery_model, acc, sparsity = lottery_ticket_round(
        lottery_model, init_state, prune_rate=0.20, epochs=5
    )
    print(f"  {rnd+1}   |  {sparsity:.1%}   |  {acc:.3f}")

## Real-World Example 2: BERT Attention Head Pruning

In [ ]:
# Simulate head-importance scoring and head pruning for a BERT-like model.
# In production, importance = mean gradient * activation over validation data.
# Here we use a synthetic importance matrix to demonstrate the pruning logic.

import numpy as np

N_LAYERS = 12
N_HEADS = 12

rng = np.random.default_rng(0)
# Synthetic head importance scores (higher = more important)
head_importance = rng.uniform(0, 1, (N_LAYERS, N_HEADS))

def prune_heads(importance: np.ndarray, keep_frac: float) -> np.ndarray:
    """Return binary mask (1=keep, 0=prune) for attention heads.
    
    Prunes globally, keeping top-keep_frac of heads by importance.
    """
    threshold = np.percentile(importance, (1 - keep_frac) * 100)
    return (importance >= threshold).astype(int)

print(f"Original: {N_LAYERS * N_HEADS} heads")
for keep in [1.0, 0.75, 0.50, 0.30]:
    mask = prune_heads(head_importance, keep_frac=keep)
    remaining = mask.sum()
    # Each head has ~64*64 params for key/query/value projections in BERT-base
    params_saved = (N_LAYERS * N_HEADS - remaining) * (64 * 64 * 3)
    print(f"Keep {keep:.0%} | Heads remaining: {remaining:3d} | "
          f"Params removed: ~{params_saved/1e6:.1f}M")

# Show which layers lose the most heads at 50% pruning
mask_50 = prune_heads(head_importance, keep_frac=0.50)
print("\nHeads kept per layer at 50% pruning:")
for layer_idx, row in enumerate(mask_50):
    bar = '#' * row.sum() + '-' * (N_HEADS - row.sum())
    print(f"  Layer {layer_idx+1:2d}: [{bar}] {row.sum()}/{N_HEADS}")

## Real-World Example 3: Sparsity vs Accuracy and Latency Trade-off

In [ ]:
# Sweep sparsity levels, record accuracy and (simulated) inference latency.
# Latency decreases only with structured pruning; unstructured needs sparse BLAS.

import time
import copy

sparsity_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []

# Train one reference model to prune from
ref_model = MLP().to(device)
quick_train(ref_model, epochs=15)

for sparsity in sparsity_levels:
    m = copy.deepcopy(ref_model)
    if sparsity > 0:
        params_to_prune = [(mod, 'weight') for mod in m.modules()
                           if isinstance(mod, nn.Linear)]
        prune.global_unstructured(
            params_to_prune, pruning_method=prune.L1Unstructured, amount=sparsity
        )
        for mod, name in params_to_prune:
            prune.remove(mod, name)

    # Accuracy
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            correct += (m(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    acc = correct / total

    # Simulate inference latency on a batch of 256
    dummy_input = torch.randn(256, 128, device=device)
    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(100):
            _ = m(dummy_input)
        latency_ms = (time.perf_counter() - t0) / 100 * 1000

    actual_sparsity = count_sparsity(m)
    results.append((actual_sparsity, acc, latency_ms))
    print(f"Sparsity {actual_sparsity:.1%} | Val Acc {acc:.3f} | "
          f"Latency {latency_ms:.2f}ms")

print("\nKey insight: accuracy holds well up to ~50% sparsity;")
print("latency reduction requires hardware sparse-BLAS support.")